# Generador de datos - Gradiente

Este notebook centraliza la generacion de datos experimentales para las funciones activas en `FUNCTION_CONFIGS`.

Alcance de este notebook:
- ejecutar corridas para los metodos configurados
- guardar resultados limpios en archivos `json` y `csv`
- guardar un resumen consolidado para `02_analisis_resultados.ipynb`

Fuera de alcance aqui:
- graficas
- histogramas
- comparaciones visuales
- animaciones


In [10]:
import csv
import json
import sys
from pathlib import Path
from typing import Any, Callable

import numpy as np
from numpy.typing import NDArray

BASE_DIR = Path.cwd()
PROJECT_DIR = BASE_DIR.parent
sys.path.insert(0, str(PROJECT_DIR))

from funciones_gradientes import (
    get_initial_position,
    goldstein_price_gradient,
    griewank_gradient,
    rastrigin_gradient,
    rosenbrock_gradient,
    run_gradient_descent,
    schwefel_gradient,
    six_hump_camel_gradient,
)
from funciones_objetivo import (
    goldstein_price,
    griewank,
    rastrigin,
    rosenbrock,
    schwefel,
    six_hump_camel,
)

Array = NDArray[np.float64]
ObjectiveFunction = Callable[[Array], float]
GradientFunction = Callable[[Array], Array]

DATA_DIR = BASE_DIR / "datos"
MANIFEST_PATH = DATA_DIR / "manifest_gradiente.json"

N_CORRIDAS_LISTA = [100, 500, 1000]
SEED_BASE = 42

FUNCTION_CONFIGS: dict[str, dict[str, Any]] = {
    "rosenbrock": {
        "display_name": "Rosenbrock",
        "dimensions": [2, 3],
        "objective_function": rosenbrock,
        "gradient_function": rosenbrock_gradient,
        "bounds": (-2.048, 2.048),
        "gradient": {
            "rate": 1e-4,
            "max_iterations": 1000,
            "tolerance": 1e-6,
        },
    },
    "schwefel": {
        "display_name": "Schwefel",
        "dimensions": [2, 3],
        "objective_function": schwefel,
        "gradient_function": schwefel_gradient,
        "bounds": (-500.0, 500.0),
        "gradient": {
            "rate": 1e-3,
            "max_iterations": 10_000,
            "tolerance": 1e-6,
        },
    },
    "rastrigin": {
        "display_name": "Rastrigin",
        "dimensions": [2, 3],
        "objective_function": rastrigin,
        "gradient_function": rastrigin_gradient,
        "bounds": (-5.12, 5.12),
        "gradient": {
            "rate": 1e-3,
            "max_iterations": 1000,
            "tolerance": 1e-6,
        },
    },
    "griewank": {
        "display_name": "Griewank",
        "dimensions": [2, 3],
        "objective_function": griewank,
        "gradient_function": griewank_gradient,
        "bounds": (-10.0, 10.0),
        "gradient": {
            "rate": 1e-2,
            "max_iterations": 2000,
            "tolerance": 1e-6,
        },
    },
    "goldstein_price": {
        "display_name": "Goldstein-Price",
        "dimensions": [2],
        "objective_function": goldstein_price,
        "gradient_function": goldstein_price_gradient,
        "bounds": (-1.5, 1.5),
        "gradient": {
            "rate": 1e-7,
            "max_iterations": 5000,
            "tolerance": 1e-6,
        },
    },
    "six_hump_camel": {
        "display_name": "Six-Hump Camel",
        "dimensions": [2],
        "objective_function": six_hump_camel,
        "gradient_function": six_hump_camel_gradient,
        "bounds": (-2.0, 2.0),
        "gradient": {
            "rate": 1e-3,
            "max_iterations": 5000,
            "tolerance": 1e-6,
        },
    },
}


## Configuracion de ejecucion

Esta celda permite correr el notebook por partes para evitar recalculos innecesarios.

Si un filtro se deja en `None`, se toman todos los valores configurados.


In [11]:
SELECTED_FUNCTIONS = None
SELECTED_DIMENSIONS = None
SELECTED_N_CORRIDAS = None
SKIP_EXISTING_FILES = True

# Ejemplos utiles:
# SELECTED_FUNCTIONS = ["rosenbrock", "schwefel", "rastrigin"]
# SELECTED_DIMENSIONS = [2, 3]
# SELECTED_N_CORRIDAS = [100, 500]
# SELECTED_FUNCTIONS = ["griewank", "goldstein_price", "six_hump_camel"]
# SELECTED_DIMENSIONS = [2]
# SELECTED_N_CORRIDAS = [100]

SELECTED_FUNCTIONS = ["griewank", "goldstein_price", "six_hump_camel"]
SELECTED_DIMENSIONS = None
SELECTED_N_CORRIDAS = [100, 500]
SKIP_EXISTING_FILES = True

In [12]:
def ensure_output_dirs() -> None:
    DATA_DIR.mkdir(parents=True, exist_ok=True)


def get_function_dir(function_name: str) -> Path:
    function_dir = DATA_DIR / function_name
    function_dir.mkdir(parents=True, exist_ok=True)
    return function_dir


def build_seed(function_name: str, method_name: str, dimension: int, run_index: int) -> int:
    key = f"{function_name}:{method_name}:{dimension}:{run_index}"
    checksum = sum(ord(char) for char in key)
    return SEED_BASE + checksum


def count_gradient_evaluations(result: dict[str, Any]) -> int:
    return len(result["function_values"]) + 1


def serialize_parameters(parameters: dict[str, Any]) -> str:
    return json.dumps(parameters, ensure_ascii=True, sort_keys=True)


def run_gradient_experiment(
    *,
    function_name: str,
    display_name: str,
    objective_function: ObjectiveFunction,
    gradient_function: GradientFunction,
    dimension: int,
    bounds: tuple[float, float],
    n_runs: int,
    parameters: dict[str, Any],
) -> list[dict[str, Any]]:
    results: list[dict[str, Any]] = []

    for run_id in range(1, n_runs + 1):
        seed = build_seed(function_name, "gradient", dimension, run_id)
        np.random.seed(seed)
        initial_position = get_initial_position(dimension, limits=bounds)

        try:
            result = run_gradient_descent(
                initial_position=initial_position,
                function=objective_function,
                gradient_function=gradient_function,
                rate=parameters["rate"],
                max_iterations=parameters["max_iterations"],
                tolerance=parameters["tolerance"],
            )
        except (OverflowError, FloatingPointError, ValueError) as error:
            print(
                f"Corrida omitida por error numerico: {function_name} | "
                f"{dimension}D | corrida={run_id} | error={error}"
            )
            continue

        final_value = float(result["final_value"])
        final_position = np.asarray(result["final_position"], dtype=float)

        if not np.isfinite(final_value) or not np.all(np.isfinite(final_position)):
            print(
                f"Corrida omitida por resultado no finito: {function_name} | "
                f"{dimension}D | corrida={run_id}"
            )
            continue

        results.append(
            {
                "funcion": function_name,
                "nombre_funcion": display_name,
                "corrida": run_id,
                "semilla": seed,
                "metodo": "gradient",
                "dimension": dimension,
                "limites": list(bounds),
                "posicion_inicial": result["initial_position"].tolist(),
                "posicion_final": final_position.tolist(),
                "valor_final": final_value,
                "iteraciones": int(result["iterations"]),
                "evaluaciones": int(count_gradient_evaluations(result)),
                "parametros": parameters.copy(),
            }
        )

    return results


def summarize_results(results: list[dict[str, Any]]) -> dict[str, Any]:
    if not results:
        raise ValueError("No hay resultados validos para resumir.")

    final_values = np.array([row["valor_final"] for row in results], dtype=float)
    evaluations = np.array([row["evaluaciones"] for row in results], dtype=float)
    iterations = np.array([row["iteraciones"] for row in results], dtype=float)

    return {
        "n_corridas": len(results),
        "mejor_valor_final": float(final_values.min()),
        "peor_valor_final": float(final_values.max()),
        "promedio_valor_final": float(final_values.mean()),
        "mediana_valor_final": float(np.median(final_values)),
        "desviacion_valor_final": float(final_values.std(ddof=0)),
        "promedio_evaluaciones": float(evaluations.mean()),
        "mediana_evaluaciones": float(np.median(evaluations)),
        "promedio_iteraciones": float(iterations.mean()),
        "mediana_iteraciones": float(np.median(iterations)),
    }


def save_run_results(
    *,
    function_name: str,
    method_name: str,
    dimension: int,
    n_runs: int,
    results: list[dict[str, Any]],
) -> tuple[Path, Path]:
    base_name = f"{function_name}_{method_name}_{dimension}d_n{n_runs}"
    function_dir = get_function_dir(function_name)
    json_path = function_dir / f"{base_name}.json"
    csv_path = function_dir / f"{base_name}.csv"

    with json_path.open("w", encoding="utf-8") as json_file:
        json.dump(results, json_file, indent=2, ensure_ascii=False)

    csv_rows = []
    for row in results:
        csv_rows.append(
            {
                "funcion": row["funcion"],
                "nombre_funcion": row["nombre_funcion"],
                "corrida": row["corrida"],
                "semilla": row["semilla"],
                "metodo": row["metodo"],
                "dimension": row["dimension"],
                "limites": json.dumps(row["limites"], ensure_ascii=True),
                "posicion_inicial": json.dumps(row["posicion_inicial"], ensure_ascii=True),
                "posicion_final": json.dumps(row["posicion_final"], ensure_ascii=True),
                "valor_final": row["valor_final"],
                "iteraciones": row["iteraciones"],
                "evaluaciones": row["evaluaciones"],
                "parametros": serialize_parameters(row["parametros"]),
            }
        )

    with csv_path.open("w", newline="", encoding="utf-8") as csv_file:
        writer = csv.DictWriter(
            csv_file,
            fieldnames=[
                "funcion",
                "nombre_funcion",
                "corrida",
                "semilla",
                "metodo",
                "dimension",
                "limites",
                "posicion_inicial",
                "posicion_final",
                "valor_final",
                "iteraciones",
                "evaluaciones",
                "parametros",
            ],
        )
        writer.writeheader()
        writer.writerows(csv_rows)

    return json_path, csv_path


def output_paths(function_name: str, method_name: str, dimension: int, n_runs: int) -> tuple[Path, Path]:
    base_name = f"{function_name}_{method_name}_{dimension}d_n{n_runs}"
    function_dir = get_function_dir(function_name)
    return function_dir / f"{base_name}.json", function_dir / f"{base_name}.csv"


def should_run(function_name: str, dimension: int, n_runs: int) -> bool:
    if SELECTED_FUNCTIONS is not None and function_name not in SELECTED_FUNCTIONS:
        return False
    if SELECTED_DIMENSIONS is not None and dimension not in SELECTED_DIMENSIONS:
        return False
    if SELECTED_N_CORRIDAS is not None and n_runs not in SELECTED_N_CORRIDAS:
        return False
    return True


def files_already_exist(function_name: str, dimension: int, n_runs: int) -> bool:
    json_path, csv_path = output_paths(function_name, "gradient", dimension, n_runs)
    return json_path.exists() and csv_path.exists()


def save_function_summary(function_name: str, summary_rows: list[dict[str, Any]]) -> tuple[Path, Path]:
    function_dir = get_function_dir(function_name)
    json_path = function_dir / f"resumen_{function_name}.json"
    csv_path = function_dir / f"resumen_{function_name}.csv"

    with json_path.open("w", encoding="utf-8") as json_file:
        json.dump(summary_rows, json_file, indent=2, ensure_ascii=False)

    with csv_path.open("w", newline="", encoding="utf-8") as csv_file:
        writer = csv.DictWriter(
            csv_file,
            fieldnames=[
                "funcion",
                "nombre_funcion",
                "metodo",
                "dimension",
                "n_corridas",
                "mejor_valor_final",
                "peor_valor_final",
                "promedio_valor_final",
                "mediana_valor_final",
                "desviacion_valor_final",
                "promedio_evaluaciones",
                "mediana_evaluaciones",
                "promedio_iteraciones",
                "mediana_iteraciones",
            ],
        )
        writer.writeheader()
        writer.writerows(summary_rows)

    return json_path, csv_path


def build_manifest() -> dict[str, Any]:
    return {
        "modulo": "gradiente",
        "funciones": list(FUNCTION_CONFIGS.keys()),
        "consumer_notebook": "analisis_resultados.ipynb",
        "descripcion": "Datos generados sin graficas ni animaciones.",
        "filtros_ejecucion": {
            "selected_functions": SELECTED_FUNCTIONS,
            "selected_dimensions": SELECTED_DIMENSIONS,
            "selected_n_corridas": SELECTED_N_CORRIDAS,
            "skip_existing_files": SKIP_EXISTING_FILES
        },
        "directorios": {
            "datos_base": "gradiente/datos"
        },
        "organizacion_corridas": "gradiente/datos/<funcion>/<archivo>.json|csv",
        "organizacion_resumenes": "gradiente/datos/<funcion>/resumen_<funcion>.json|csv",
        "campos_corridas": [
            "funcion",
            "nombre_funcion",
            "corrida",
            "semilla",
            "metodo",
            "dimension",
            "limites",
            "posicion_inicial",
            "posicion_final",
            "valor_final",
            "iteraciones",
            "evaluaciones",
            "parametros"
        ],
        "campos_resumen": [
            "funcion",
            "nombre_funcion",
            "metodo",
            "dimension",
            "n_corridas",
            "mejor_valor_final",
            "peor_valor_final",
            "promedio_valor_final",
            "mediana_valor_final",
            "desviacion_valor_final",
            "promedio_evaluaciones",
            "mediana_evaluaciones",
            "promedio_iteraciones",
            "mediana_iteraciones"
        ]
    }


def save_manifest() -> Path:
    manifest = build_manifest()
    with MANIFEST_PATH.open("w", encoding="utf-8") as json_file:
        json.dump(manifest, json_file, indent=2, ensure_ascii=False)
    return MANIFEST_PATH


def print_summary_row(row: dict[str, Any]) -> None:
    print(
        f"{row['funcion']} | {row['metodo']} | {row['dimension']}D | "
        f"n={row['n_corridas']} | mejor={row['mejor_valor_final']:.8f} | "
        f"promedio={row['promedio_valor_final']:.8f} | "
        f"eval_prom={row['promedio_evaluaciones']:.2f}"
    )


In [13]:
ensure_output_dirs()

generated_anything = False
generated_labels: list[str] = []
function_summary_rows: dict[str, list[dict[str, Any]]] = {name: [] for name in FUNCTION_CONFIGS}

for function_name, config in FUNCTION_CONFIGS.items():
    for dimension in config["dimensions"]:
        for n_runs in N_CORRIDAS_LISTA:
            if should_run(function_name, dimension, n_runs):
                if SKIP_EXISTING_FILES and files_already_exist(function_name, dimension, n_runs):
                    print(f"Saltando existente: {function_name} | gradient | {dimension}D | n={n_runs}")
                else:
                    gradient_results = run_gradient_experiment(
                        function_name=function_name,
                        display_name=config["display_name"],
                        objective_function=config["objective_function"],
                        gradient_function=config["gradient_function"],
                        dimension=dimension,
                        bounds=config["bounds"],
                        n_runs=n_runs,
                        parameters=config["gradient"],
                    )
                    save_run_results(
                        function_name=function_name,
                        method_name="gradient",
                        dimension=dimension,
                        n_runs=n_runs,
                        results=gradient_results,
                    )
                    if not gradient_results:
                        print(
                            f"Sin resultados validos: {function_name} | gradient | {dimension}D | n={n_runs}"
                        )
                        continue

                    gradient_summary = {
                        "funcion": function_name,
                        "nombre_funcion": config["display_name"],
                        "metodo": "gradient",
                        "dimension": dimension,
                        **summarize_results(gradient_results),
                    }
                    generated_anything = True
                    generated_labels.append(f"{function_name} | gradient | {dimension}D | n={n_runs}")
                    function_summary_rows[function_name].append(gradient_summary)
                    print_summary_row(gradient_summary)

if generated_anything:
    print()
    print("Guardando resumenes por funcion...")
    for function_name, rows in function_summary_rows.items():
        if rows:
            summary_json, summary_csv = save_function_summary(function_name, rows)
            print(f"- {function_name}: {summary_json.name}, {summary_csv.name}")
    print()
    print("Corridas nuevas generadas:")
    for label in generated_labels:
        print(f"- {label}")
else:
    print("No hubo nuevas corridas en esta ejecucion.")

manifest_path = save_manifest()

print()
print(f"Manifest guardado en: {manifest_path}")


griewank | gradient | 2D | n=100 | mejor=0.00000001 | promedio=0.01550731 | eval_prom=1766.94
griewank | gradient | 2D | n=500 | mejor=0.00000001 | promedio=0.02001996 | eval_prom=1732.03
griewank | gradient | 3D | n=100 | mejor=0.00739860 | promedio=0.03749858 | eval_prom=2002.00
griewank | gradient | 3D | n=500 | mejor=0.00739860 | promedio=0.03517385 | eval_prom=2002.00


c:\Carlos\Uni\Algortimos\trabajos\trabajo_1\trabajo_carlos\1. optimizacion_numerica\gradiente\funciones_objetivo.py:61: RuntimeWarning: overflow encountered in scalar multiply
  return first_term * second_term
c:\Carlos\Uni\Algortimos\trabajos\trabajo_1\trabajo_carlos\1. optimizacion_numerica\gradiente\funciones_objetivo.py:54: RuntimeWarning: overflow encountered in power
  first_term = 1 + np.power(x1 + x2 + 1, 2) * (


OverflowError: (34, 'Result too large')